# ❤️ Proyecto: Predicción de Enfermedades Cardiovasculares

## 🎯 Objetivos del Proyecto

En este proyecto práctico aprenderás a:

1. **Trabajar con datos médicos reales** del UCI Heart Disease Dataset
2. **Construir un pipeline completo** de Machine Learning
3. **Evaluar modelos de clasificación binaria** con múltiples métricas
4. **Comparar diferentes algoritmos** (SGD vs Random Forest)
5. **Registrar experimentos** con MLflow de forma profesional
6. **Interpretar resultados médicos** con responsabilidad

---

## ❤️ ¿Por qué es importante?

Las **Enfermedades Cardiovasculares (ECV)** son la principal causa de muerte en el mundo:

- 💔 Causan **17.9 millones** de muertes al año
- ⚠️ **90% son prevenibles** con detección temprana
- 🏥 Un diagnóstico temprano puede **salvar vidas**
- 🤖 La IA puede ayudar a **detectar patrones** que salven vidas

---

## 📊 El Dataset: UCI Heart Disease

- **303 pacientes** con datos clínicos
- **14 características** médicas (edad, presión arterial, colesterol, etc.)
- **Variable objetivo**: Presencia de enfermedad cardíaca (0/1)
- **Tipo de problema**: Clasificación binaria

---

## 🚀 ¡Empecemos!

## ⚙️ Paso 2: Configuración de MLflow

## 📚 Paso 3: Importar Librerías

In [0]:

%pip install seaborn
dbutils.library.restartPython()

In [0]:
import mlflow
import warnings
warnings.filterwarnings('ignore')

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks")

# Debo sustituir esto por mi propio email de Databricks (el mismo con el que inicio sesión).
# Se usa para construir la ruta del experimento en /Users/<mi-email>/..., donde tengo
# permisos de escritura. Si lo dejo vacío, la validación de abajo avisa y no se configura
# el experimento.
email = 'gonzalovizoso97@gmail.com'  # ⚠️ CAMBIAR POR TU EMAIL DE DATABRICKS

if not email:
    print("⚠️  ADVERTENCIA: Debes configurar tu email antes de continuar")
else:
    # set_tracking_uri le dice a MLflow dónde guardar y consultar los runs (parámetros,
    # métricas, modelos). Con "databricks" usa el servidor de tracking gestionado por
    # Databricks en vez de guardar los datos en local.
    mlflow.set_tracking_uri("databricks")

    # set_experiment selecciona (o crea si no existe) el experimento donde se agruparán
    # todos los runs que ejecute a continuación, en vez de mezclarse con otros proyectos.
    experiment_name = f"/Users/{email}/5-prediccion-infarto"
    mlflow.set_experiment(experiment_name)

    print("=" * 70)
    print("✅ MLflow configurado correctamente")
    print("=" * 70)
    print(f"📊 Experimento: {experiment_name}")
    print(f"❤️  Proyecto: Predicción de Enfermedades Cardiovasculares")
    print("=" * 70)

In [0]:
# Librerías básicas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# train_test_split divide un dataset en dos subconjuntos (entrenamiento y prueba) de
# forma aleatoria, manteniendo la proporción indicada. Necesario para evaluar el modelo
# con datos que no ha visto durante el entrenamiento.
from sklearn.model_selection import train_test_split

# Un Pipeline encadena varios pasos de transformación de datos (y opcionalmente un
# modelo al final) en una sola secuencia. Evita aplicar cada paso a mano, reduce errores
# y asegura que a train y test se les aplican exactamente las mismas transformaciones.
from sklearn.pipeline import Pipeline

# Preprocesamiento
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# SGDClassifier: modelo lineal simple entrenado por descenso de gradiente estocástico,
# usado como baseline rápido y sencillo.
# RandomForestClassifier: conjunto (ensemble) de árboles de decisión que promedian sus
# predicciones, capaz de capturar relaciones no lineales.
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier

# cross_val_score evalúa un modelo con validación cruzada: divide el train en K folds,
# entrena K veces usando K-1 folds y valida con el restante, dando una estimación más
# fiable que un único split train/test.
from sklearn.model_selection import cross_val_score, cross_val_predict

# Métricas de clasificación binaria:
# precision_score: de los predichos como enfermos, cuántos lo están de verdad.
# recall_score: de los enfermos reales, cuántos detecta el modelo (clave en medicina).
# f1_score: media armónica entre precision y recall.
# confusion_matrix / ConfusionMatrixDisplay: desglose de aciertos y errores por clase.
# roc_auc_score: qué tan bien separa el modelo las clases en todos los umbrales.
# accuracy_score: proporción total de aciertos.
# classification_report: tabla resumen con todas las métricas por clase.
# roc_curve: puntos para dibujar la curva ROC.
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    accuracy_score,
    classification_report,
    roc_curve
)

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")

print("✅ Todas las librerías importadas correctamente")

## ❤️ Contexto del Proyecto

### El Problema

En este proyecto trabajaremos con el **UCI Heart Disease Dataset**, uno de los datasets
más importantes en medicina predictiva.

### 🎯 Nuestro Objetivo

Construir un modelo de Machine Learning que pueda **predecir la presencia de enfermedad
cardiovascular** basándose en características clínicas del paciente.

### ⚕️ Responsabilidad Ética

> ⚠️ **IMPORTANTE**: Este es un proyecto educativo. Los modelos médicos reales requieren
> validación clínica exhaustiva, aprobación regulatoria, supervisión médica profesional
> y consideraciones éticas y legales. Nuestro objetivo es aprender las técnicas, no
> reemplazar el criterio médico.

## 📚 Fundamentos: Clasificación Binaria

### 🎯 ¿Qué es Clasificación Binaria?

Un problema donde el modelo debe elegir entre **dos clases**:
- ✅ Clase Positiva (1): Tiene enfermedad cardíaca
- ❌ Clase Negativa (0): No tiene enfermedad cardíaca

### 📊 Métricas de Evaluación

| Situación | Métrica Principal |
|-----------|-------------------|
| Clases balanceadas | Accuracy |
| No perder positivos (medicina) | **Recall** ⭐ |
| Evitar falsos positivos | Precision |
| Balance general | F1-Score |
| Comparar modelos | ROC-AUC |

**En medicina, típicamente priorizamos Recall** porque es mejor detectar un caso falso
positivo que perder un verdadero positivo.

## 📁 Paso 4: Cargar y Explorar los Datos

In [0]:
import os
print(os.getcwd())


In [0]:
# El archivo heart.csv contiene 303 pacientes con 13 características clínicas
# (demográficas, síntomas y mediciones médicas) y una columna 'target' que indica si el
# paciente tiene enfermedad cardíaca (1) o no (0). Lo cargo desde data/raw del repo.
# Si esta ruta da error en tu Git folder, ajústala según dónde tengas clonado el repo.
data = pd.read_csv("../data/raw/heart.csv")
print(f"Filas: {data.shape[0]}")
print(f"Columnas: {data.shape[1]}")
print(f"Variable objetivo: 'target' (0 = no enfermedad, 1 = enfermedad)")

In [0]:
# head() permite ver cómo están estructurados los datos: qué tipo de valores tiene cada
# columna y detectar a simple vista algo raro antes de analizar más a fondo.
data.head()

### 📋 Diccionario de Datos

#### 👤 Datos Demográficos
1. **age**: Edad del paciente en años
2. **sex**: Sexo (1 = masculino, 0 = femenino)

#### 💊 Síntomas y Diagnóstico
3. **cp**: Tipo de dolor de pecho (0: angina típica, 1: angina atípica, 2: dolor no anginoso, 3: asintomático)
4. **exang**: Angina inducida por ejercicio (1 = sí, 0 = no)

#### 🩺 Mediciones Clínicas
5. **trestbps**: Presión arterial en reposo (mm Hg)
6. **chol**: Colesterol sérico (mg/dl)
7. **fbs**: Azúcar en sangre en ayunas > 120 mg/dl (1 = verdadero, 0 = falso)
8. **thalach**: Frecuencia cardíaca máxima alcanzada

#### 📊 Resultados de Pruebas
9. **restecg**: Resultados electrocardiográficos en reposo (0: normal, 1: anormalidad ST-T, 2: hipertrofia ventricular izquierda)
10. **oldpeak**: Depresión del segmento ST inducida por ejercicio
11. **slope**: Pendiente del segmento ST durante ejercicio (0: ascendente, 1: plano, 2: descendente)
12. **ca**: Número de vasos principales coloreados por fluoroscopia (0-3)
13. **thal**: Resultados de prueba de talasemia (1: normal, 2: defecto fijo, 3: defecto reversible)

#### 🎯 Variable Objetivo
14. **target**: Presencia de enfermedad cardíaca (0: no, 1: sí)

In [0]:
# info() muestra los tipos de dato de cada columna y si hay valores nulos.
data.info()

# Columnas categóricas (aunque estén codificadas como números): sex, cp, fbs, restecg,
# exang, slope, ca, thal. Son códigos que representan categorías, no cantidades continuas.
# Columnas numéricas (continuas): age, trestbps, chol, thalach, oldpeak.
# 'target' es la variable objetivo, no una característica de entrada.

In [0]:
# describe() da estadísticas descriptivas (media, min, max, cuartiles) de cada columna
# numérica. Ejecuta esta celda y mira TUS valores reales antes de escribir observaciones.
data.describe()

### 💡 Observaciones Clave

- La edad media de los pacientes es de 54.4 años, con un rango de 29 a 77 años.
- La presión arterial en reposo (`trestbps`) tiene una media de 131.5 mm Hg, con un
  máximo de 200 mm Hg, un valor claramente elevado (hipertensión).
- El colesterol (`chol`) tiene una media de 245.9 mg/dl y un máximo de 564 mg/dl. Ese
  máximo está muy por encima del resto de la distribución (el percentil 75 es 275), lo
  que sugiere la presencia de algún valor atípico.
- `thalach` (frecuencia cardíaca máxima) va de 71 a 202.
- A diferencia de lo que esperaba, `trestbps` y `chol` **sí tienen valores nulos**:
  258 y 273 registros no nulos de 303 totales, respectivamente (45 y 30 valores
  faltantes). Esto confirma que el `SimpleImputer` del pipeline de preprocesamiento es
  necesario, no solo una buena práctica preventiva.
- Las escalas de las variables numéricas son muy distintas entre sí (edad y presión en
  decenas, colesterol en cientos, `oldpeak` en unidades), lo que justifica
  normalizarlas antes de entrenar modelos sensibles a la escala, como SGD.

In [0]:
# Distribución de la variable objetivo: cuántos pacientes hay de cada clase.
target_counts = data.target.value_counts()
print(target_counts)

# Proporciones por clase.
target_props = data.target.value_counts(normalize=True)
print(target_props)

# Distribución: 165 pacientes con enfermedad (target=1) y 138 sin enfermedad (target=0),
# de un total de 303. En proporción: 54.5% clase 1 y 45.5% clase 0.
#
# El dataset está razonablemente balanceado: la clase mayoritaria (1) no llega ni al
# 55%, muy lejos del 70-80% que marcaría un desbalance problemático. Esto importa
# porque, si una clase dominara con claridad, un modelo "tonto" que predijera siempre
# esa clase obtendría un accuracy engañosamente alto sin haber aprendido nada. Al estar
# balanceado, el accuracy es una métrica razonable de referencia, aunque en este
# proyecto seguimos priorizando el recall por las implicaciones clínicas de un falso
# negativo.

### 🔍 Análisis Exploratorio Visual

In [0]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

sns.histplot(data=data, x='age', hue='target', multiple='stack', ax=axes[0, 0])
axes[0, 0].set_title('Distribución de edad por diagnóstico')

sns.scatterplot(data=data, x='chol', y='trestbps', hue='target', alpha=0.7, ax=axes[0, 1])
axes[0, 1].set_title('Colesterol vs presión arterial por diagnóstico')

sns.countplot(data=data, x='cp', hue='target', ax=axes[1, 0])
axes[1, 0].set_title('Tipo de dolor de pecho vs diagnóstico')

sns.heatmap(data.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1, 1])
axes[1, 1].set_title('Matriz de correlación (solo variables numéricas)')

plt.tight_layout()
plt.show()

# Observaciones:
# - En el histograma de edad, los pacientes con enfermedad (target=1, naranja) se
#   concentran algo más en el tramo de 40-55 años, mientras que los sanos (target=0,
#   verde) predominan a partir de los 55-60. Hay bastante solapamiento entre ambos
#   grupos en todo el rango, así que la edad por sí sola no separa bien las clases.
# - En el scatter de colesterol vs presión arterial, los colores (target) están
#   completamente mezclados, sin ninguna zona donde predomine un color sobre otro. Estas
#   dos variables, combinadas solo entre sí, no separan a los pacientes enfermos de los
#   sanos.
# - En el countplot de tipo de dolor de pecho, la relación es muy clara: 'typical
#   angina' es la categoría más común, pero la mayoría de esos pacientes NO tienen
#   enfermedad (target=0). En cambio, 'non-anginal pain' y 'atypical angina' están
#   claramente dominadas por pacientes con enfermedad (target=1). Es contraintuitivo,
#   pero así lo muestran los datos, y sugiere que 'cp' es una variable muy relevante
#   para el modelo.
# - En la matriz de correlación, las variables con mayor correlación (en valor
#   absoluto) con 'target' son 'oldpeak' (-0.43), 'ca' (-0.39), 'thalach' (0.42) y
#   'thal' (-0.34). Es decir: cuanto mayor es la frecuencia cardíaca máxima alcanzada
#   (thalach), más probable es la enfermedad; mientras que a mayor depresión del
#   segmento ST (oldpeak) y mayor número de vasos coloreados por fluoroscopia (ca),
#   menos probable aparece la enfermedad en esta codificación del target. El colesterol
#   (chol, -0.07) y la presión arterial (trestbps, -0.12) apenas correlacionan de forma
#   lineal con target, coincidiendo con lo observado en el scatter de arriba.

## 🔀 Paso 5: Dividir los Datos

In [0]:
# Divido en train (80%) y test (20%). stratify=data['target'] mantiene la misma
# proporción de enfermos/sanos en ambos conjuntos. random_state fija la semilla para
# que el split sea reproducible.
train_set, test_set = train_test_split(
    data, test_size=0.2, random_state=42, stratify=data['target']
)

print(f"Train: {train_set.shape[0]} filas")
print(f"Test: {test_set.shape[0]} filas")
print(train_set['target'].value_counts(normalize=True))
print(test_set['target'].value_counts(normalize=True))

In [0]:
train_set.hist(bins=50, figsize=(20, 15))
plt.show()

# Características con escalas muy distintas:
# 'chol' va de ~150 a >400 y 'trestbps' de ~100 a 200, ambas en rangos de cientos, frente
# a 'oldpeak' que se mueve entre 0 y ~6.2, un rango de unidades. 'age' y 'thalach' están
# en decenas. Esta diferencia de escalas confirma que hace falta estandarizar antes de
# entrenar modelos como SGD, que son sensibles a la magnitud de cada variable.
#
# También se aprecia que:
# - 'oldpeak' tiene una distribución muy asimétrica (sesgada a la derecha): la mayoría
#   de pacientes tiene valores cercanos a 0, con una cola larga hacia valores altos.
# - 'ca' y 'thal', aunque numéricas, se comportan como variables discretas/categóricas
#   (pocos valores posibles con barras separadas), coherente con que estén en cat_attr
#   más adelante en vez de tratarse como continuas.
# - 'target' confirma el balance ya visto: dos barras de tamaño similar (138 vs 165).

### 📏 Observación Importante: Escalas Diferentes

- Es un problema porque modelos como SGD o KNN calculan distancias o gradientes
  basados directamente en los valores numéricos. Si una variable como el colesterol
  (rango de cientos) convive con otra como oldpeak (rango de unidades), la primera
  dominará el cálculo simplemente por tener números más grandes, no porque sea más
  importante para el diagnóstico.

- Aplicaría un StandardScaler, que resta la media y divide por la desviación
  estándar de cada variable, dejándolas todas centradas en 0 y con varianza 1.

- Esto ayuda porque pone a todas las variables en pie de igualdad: el modelo aprende
  según la relación real de cada variable con el target, no según su escala original.
  Modelos basados en árboles (como Random Forest) no lo necesitan, porque dividen el
  espacio comparando una variable a la vez contra un umbral, y ahí da igual la escala.

## 🔧 Paso 6: Construir el Pipeline de Preprocesamiento

In [0]:
# Clasifico las columnas en categóricas y numéricas.
cat_attr = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
num_attr = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

# Pipeline para variables numéricas: imputo posibles nulos con la mediana (más robusta
# a valores atípicos que la media) y estandarizo con StandardScaler.
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

# ColumnTransformer combina el pipeline numérico con One-Hot Encoding para las
# categóricas, y las une en una sola matriz de salida.
full_pipeline = ColumnTransformer([
    ('num', num_pipeline, num_attr),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_attr),
])

# Uso One-Hot Encoding para las categóricas porque son códigos (0, 1, 2, 3) que
# representan categorías sin un orden ni magnitud real entre ellas. Si las dejara como
# números, el modelo podría interpretar que la categoría 3 es "más" que la 1, algo sin
# sentido clínico. One-Hot crea una columna binaria por categoría, tratándolas como
# opciones independientes.

## 🎯 Paso 7: Preparar X e y

In [0]:
x_train = train_set.drop('target', axis=1)
y_train = train_set['target']

print(f"x_train: {x_train.shape}")
print(f"y_train: {y_train.shape}")

In [0]:
# fit_transform() aprende del conjunto de datos (media/desviación para el scaler, qué
# categorías existen para el OneHotEncoder) y aplica la transformación. Solo debe usarse
# sobre train.
x_train_pr = full_pipeline.fit_transform(x_train)

print(f"Columnas antes del pipeline: {x_train.shape[1]}")
print(f"Columnas después del pipeline: {x_train_pr.shape[1]}")
# El número de columnas aumenta porque cada variable categórica se expande en varias
# columnas binarias (una por categoría), mientras que las numéricas se mantienen en una
# columna cada una, solo que reescaladas.

## 🤖 Paso 8: Entrenamiento y Evaluación de Modelos

### 🔵 Modelo 1: SGD Classifier (Baseline)

SGD (Stochastic Gradient Descent, descenso de gradiente estocástico) es un método de
optimización que ajusta los parámetros de un modelo lineal poco a poco, actualizándolos
con cada muestra (o pequeños lotes) en vez de usar todo el dataset de una vez.
`SGDClassifier` lo usa para entrenar un clasificador lineal: en cada paso calcula el
error de la predicción y ajusta los pesos en la dirección que más lo reduce. Es rápido
y escala bien, pero al ser lineal solo puede separar las clases con una frontera recta,
por lo que aquí lo usamos como referencia simple antes de un modelo más potente.

In [0]:
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="SGD Classifier - Baseline") as run:
    sgd_clf = SGDClassifier(random_state=42)

    scores = cross_val_score(sgd_clf, x_train_pr, y_train, cv=5, scoring="accuracy")

    mlflow.log_metric("cv_accuracy_mean", scores.mean())
    mlflow.log_metric("cv_accuracy_std", scores.std())

    print(f"Accuracy por fold: {scores.round(3)}")
    print(f"CV accuracy: {scores.mean():.3f} ± {scores.std():.3f}")
    # Como baseline, un modelo lineal simple da una idea de qué accuracy es alcanzable
    # sin mucho esfuerzo. Sirve de referencia mínima: un modelo más complejo debería
    # superar claramente este resultado para justificar su uso.

### 📊 Evaluación del Modelo SGD

In [0]:
# cross_val_score solo da el accuracy resumido de cada fold, sin predicciones
# individuales. cross_val_predict entrena igual que la validación cruzada, pero
# devuelve la predicción concreta de cada muestra cuando estuvo en el fold de
# validación (nunca cuando fue usada para entrenar ese modelo). Así obtengo un vector
# de predicciones "honesto" para las muestras de train, sin tocar el test, y puedo
# calcular con él la matriz de confusión y otras métricas.
sgd_preds = cross_val_predict(sgd_clf, x_train_pr, y_train, cv=5)

In [0]:
cm = confusion_matrix(y_train, sgd_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No enfermedad", "Enfermedad"])
disp.plot(cmap="Blues")
plt.title("Matriz de confusión - SGD Classifier")
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"TN (verdaderos negativos): {tn} — pacientes sanos correctamente identificados")
print(f"FP (falsos positivos): {fp} — pacientes sanos clasificados como enfermos")
print(f"FN (falsos negativos): {fn} — pacientes enfermos clasificados como sanos")
print(f"TP (verdaderos positivos): {tp} — pacientes enfermos correctamente identificados")

# El falso negativo (FN) es el error más grave en medicina: significa decirle a un
# paciente enfermo que está sano, por lo que no recibiría el seguimiento o tratamiento
# necesario. Un falso positivo (FP) también tiene coste (pruebas adicionales
# innecesarias), pero es mucho menos peligroso que dejar pasar una enfermedad real.

In [0]:
precision = precision_score(y_train, sgd_preds)
recall = recall_score(y_train, sgd_preds)
f1 = f1_score(y_train, sgd_preds)
roc_auc = roc_auc_score(y_train, sgd_preds)

print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-score: {f1:.3f}")
print(f"ROC-AUC: {roc_auc:.3f}")

# El recall es la métrica más importante en este contexto: mide qué proporción de los
# pacientes realmente enfermos detecta el modelo. Un recall bajo implica muchos falsos
# negativos, el error más costoso en un escenario médico.

### 🌲 Modelo 2: Random Forest Classifier

Random Forest es un modelo de ensemble que construye muchos árboles de decisión
distintos (cada uno con una muestra aleatoria de los datos y un subconjunto aleatorio
de variables en cada división) y combina sus predicciones, normalmente por voto
mayoritario. Esta aleatoriedad hace que los árboles cometan errores distintos entre sí,
y al promediarlos se compensan, reduciendo la varianza que tendría un único árbol. Por
eso suele generalizar mejor que un modelo lineal simple: puede capturar relaciones no
lineales e interacciones entre variables.

In [0]:
with mlflow.start_run(run_name="Random Forest Classifier") as run:
    rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)

    rf_scores = cross_val_score(rf_clf, x_train_pr, y_train, cv=5, scoring="accuracy")
    rf_preds = cross_val_predict(rf_clf, x_train_pr, y_train, cv=5)

    mlflow.log_metric("cv_accuracy_mean", rf_scores.mean())
    mlflow.log_metric("cv_accuracy_std", rf_scores.std())

    print(f"CV accuracy: {rf_scores.mean():.3f} ± {rf_scores.std():.3f}")

# Hiperparámetros que podría ajustar: n_estimators (número de árboles), max_depth
# (profundidad máxima, para controlar overfitting) y max_features (cuántas variables
# considera en cada división). Comparo contra SGD porque es mi modelo de referencia: si
# Random Forest no mejora claramente sus métricas, no estaría justificado usar un
# modelo más complejo y costoso solo por ser "más avanzado".

In [0]:
cm_rf = confusion_matrix(y_train, rf_preds)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=["No enfermedad", "Enfermedad"])
disp_rf.plot(cmap="Greens")
plt.title("Matriz de confusión - Random Forest")
plt.show()

tn_rf, fp_rf, fn_rf, tp_rf = cm_rf.ravel()
print(f"SGD  -> FN: {fn}, FP: {fp}")
print(f"RF   -> FN: {fn_rf}, FP: {fp_rf}")
# Comparo especialmente los falsos negativos de ambos modelos, porque es el error más
# grave en este problema. Si Random Forest reduce los FN respecto a SGD, es una señal
# de que generaliza mejor para el objetivo que más importa aquí.

In [0]:
rf_precision = precision_score(y_train, rf_preds)
rf_recall = recall_score(y_train, rf_preds)
rf_f1 = f1_score(y_train, rf_preds)
rf_roc_auc = roc_auc_score(y_train, rf_preds)

print(f"Precision: {rf_precision:.3f}")
print(f"Recall: {rf_recall:.3f}")
print(f"F1-score: {rf_f1:.3f}")
print(f"ROC-AUC: {rf_roc_auc:.3f}")

comparativa = pd.DataFrame({
    "Modelo": ["SGD", "Random Forest"],
    "Precision": [precision, rf_precision],
    "Recall": [recall, rf_recall],
    "F1-score": [f1, rf_f1],
    "ROC-AUC": [roc_auc, rf_roc_auc],
})
display(comparativa)

# Conclusión:
# Random Forest supera a SGD en las cuatro métricas: precision (0.858 vs 0.839),
# recall (0.871 vs 0.788), F1-score (0.865 vs 0.8125) y ROC-AUC (0.849 vs 0.803).
#
# La diferencia más relevante para este problema es la de recall: Random Forest detecta
# el 87.1% de los pacientes enfermos reales, frente al 78.8% de SGD. En un contexto
# médico donde el falso negativo es el error más grave, esos casi 9 puntos porcentuales
# de recall representan enfermos que SGD dejaría pasar y Random Forest sí detecta.
#
# El ROC-AUC (0.849 frente a 0.803) confirma que Random Forest separa mejor ambas
# clases en general, no solo en el umbral de decisión por defecto. Por todo esto,
# continuamos con Random Forest como modelo elegido para el entrenamiento final.

## 🎯 Paso 9: Entrenamiento Final y Evaluación en Test

In [0]:
forest_clf = RandomForestClassifier(n_estimators=100, random_state=42)
forest_clf.fit(x_train_pr, y_train)

# Durante la validación cruzada, cada modelo se entrenó solo con una parte del train
# (4 de los 5 folds) para poder validar con el fold restante. Una vez decidido qué
# modelo usar, aprovecho el 100% de los datos de entrenamiento disponibles para
# entrenar la versión final. El test, que nunca se ha usado hasta ahora, se reserva
# para la evaluación final.

In [0]:
x_test = test_set.drop('target', axis=1)
y_test = test_set['target']

print(f"x_test: {x_test.shape}")
print(f"y_test: {y_test.shape}")

In [0]:
# Uso transform() (no fit_transform()) porque el pipeline ya aprendió del train (media,
# desviación, categorías). Si recalculara con el test, filtraría información de esos
# datos hacia el preprocesamiento (data leakage) y la evaluación no sería realista.
x_test_pr = full_pipeline.transform(x_test)
final_preds = forest_clf.predict(x_test_pr)

In [0]:
final_precision = precision_score(y_test, final_preds)
final_recall = recall_score(y_test, final_preds)
final_f1 = f1_score(y_test, final_preds)
final_roc_auc = roc_auc_score(y_test, final_preds)
final_accuracy = accuracy_score(y_test, final_preds)

print(f"Accuracy:  {final_accuracy:.3f}")
print(f"Precision: {final_precision:.3f}")
print(f"Recall:    {final_recall:.3f}")
print(f"F1-score:  {final_f1:.3f}")
print(f"ROC-AUC:   {final_roc_auc:.3f}")

cm_final = confusion_matrix(y_test, final_preds)
disp_final = ConfusionMatrixDisplay(confusion_matrix=cm_final, display_labels=["No enfermedad", "Enfermedad"])
disp_final.plot(cmap="Purples")
plt.title("Matriz de confusión - Modelo final (test)")
plt.show()

print(classification_report(y_test, final_preds, target_names=["No enfermedad", "Enfermedad"]))

# Interpretación:
# En test (61 pacientes: 28 sin enfermedad, 33 con enfermedad), el modelo obtiene
# accuracy 0.852, precision 0.816, recall 0.939, F1 0.873 y ROC-AUC 0.845.
#
# Comparado con la validación cruzada de Random Forest sobre train (precision 0.858,
# recall 0.871, F1 0.865, ROC-AUC 0.849), los resultados son muy consistentes: el
# ROC-AUC es prácticamente idéntico (0.845 vs 0.849) y el F1 también (0.873 vs 0.865).
# No hay caída de rendimiento en test, así que el modelo generaliza bien y no muestra
# overfitting hacia el conjunto de entrenamiento.
#
# Es más, el recall sube en test respecto a la CV (0.939 vs 0.871): de los 33 pacientes
# enfermos del test, el modelo detecta 31 y solo deja pasar 2 (falsos negativos), como
# confirma la matriz de confusión. A cambio, la precision baja ligeramente (0.816 vs
# 0.858): comete 7 falsos positivos (pacientes sanos marcados como enfermos). Dado que
# en este problema priorizamos el recall (evitar falsos negativos es más importante que
# evitar falsos positivos), este resultado final es coherente con el objetivo del
# proyecto: solo 2 de 33 enfermos reales pasan desapercibidos.

## 🎉 Reflexión Final del Proyecto

### ✅ Comprueba que has trabajado

1. [x] Exploración de datos médicos reales
2. [x] Pipeline de preprocesamiento
3. [x] Entrenamiento y comparación de modelos
4. [x] Evaluación con métricas de clasificación
5. [x] Validación cruzada
6. [x] Registro de experimentos en MLflow
7. [x] Interpretación de resultados con matrices de confusión

### 📊 Conclusiones Clave

1. **¿Qué modelo funcionó mejor?**
   - Random Forest. En validación cruzada superó a SGD en las cuatro métricas:
     precision (0.858 vs 0.839), recall (0.871 vs 0.788), F1 (0.865 vs 0.8125) y
     ROC-AUC (0.849 vs 0.803). En el conjunto de test final, mantuvo un rendimiento
     muy similar (ROC-AUC 0.845, F1 0.873), confirmando que la mejora era real y no
     casualidad de la validación cruzada.

2. **¿Por qué crees que funcionó mejor?**
   - SGD es un modelo lineal: solo puede separar las clases con una frontera recta en
     el espacio de características. Random Forest combina muchos árboles entrenados
     con subconjuntos aleatorios de datos y variables, lo que le permite capturar
     relaciones no lineales e interacciones entre características (por ejemplo, entre
     el tipo de dolor de pecho y la frecuencia cardíaca máxima) que un modelo lineal
     no puede representar.

3. **¿El modelo es suficientemente bueno para uso médico?**
   - No, no para uso clínico real. Aunque el recall en test es alto (0.939, solo 2
     falsos negativos de 33 enfermos), el dataset tiene apenas 303 pacientes de una
     única fuente, sin validación externa en otros hospitales o poblaciones, sin
     revisión por profesionales médicos y sin aprobación regulatoria. Un modelo
     médico real necesita todo esto antes de poder usarse, aunque como ejercicio
     educativo demuestra bien la metodología.

4. **¿Qué métrica es más importante en este caso?**
   - El recall, porque el coste de un falso negativo (decirle a un enfermo que está
     sano) es mucho mayor que el de un falso positivo (alertar de más a un paciente
     sano). El modelo final logra un recall de 0.939 en test, dejando pasar solo 2
     enfermos de 33, a cambio de 7 falsos positivos sobre 28 pacientes sanos.

5. **¿Qué mejorarías del modelo?**
   - Probaría GridSearchCV para ajustar hiperparámetros de Random Forest (n_estimators,
     max_depth, min_samples_split), estudiaría la importancia de características para
     entender qué variables pesan más en la decisión, y ampliaría el dataset si fuera
     posible, ya que 303 pacientes es una muestra pequeña para un problema médico real.

### 💡 Reflexiones Importantes

#### ⚕️ Sobre Medicina e IA

1. **Falsos Negativos vs Falsos Positivos**
   - En medicina, el falso negativo es más grave. Decirle a un paciente enfermo que
     está sano retrasa un tratamiento que podría ser vital, mientras que un falso
     positivo genera pruebas adicionales y preocupación, pero no pone en riesgo
     directo la vida del paciente. En nuestro modelo final, los 2 falsos negativos
     (de 33 enfermos) son el número que más pesa a la hora de valorar si el modelo
     es aceptable, más que los 7 falsos positivos.

2. **Responsabilidad Ética**
   - No. Un modelo de IA puede servir como herramienta de apoyo, señalando casos de
     riesgo o priorizando qué pacientes revisar antes, pero la decisión final debe
     recaer en un profesional sanitario que valore el contexto completo del paciente:
     su historial, otros síntomas, y factores que el modelo no ve porque no están en
     estas 13 variables. Delegar la decisión completa en el modelo trasladaría la
     responsabilidad médica a un sistema que no puede rendir cuentas ni razonar sobre
     casos atípicos.

3. **Interpretabilidad**
   - Es importante porque los médicos necesitan poder justificar sus decisiones ante
     el paciente y ante la propia institución, y confiar en por qué el modelo hace una
     recomendación antes de actuar sobre ella. Un modelo que funciona como caja negra,
     sin poder explicar qué variables llevaron a una predicción, es más difícil de
     auditar, de corregir cuando se equivoca y de aceptar en un entorno clínico donde
     los errores tienen consecuencias serias. Random Forest, aunque menos transparente
     que un único árbol de decisión, sigue permitiendo extraer la importancia de cada
     característica (Desafío 3), lo que ayuda a mantener cierto grado de explicación.

### 🚀 Desafíos Adicionales

1. **Optimización de Hiperparámetros** con `GridSearchCV`.
2. **Prueba Otros Modelos** como Gradient Boosting, SVM o Logistic Regression.
3. **Feature Engineering** para crear o seleccionar características.
4. **Análisis de Errores** para entender pacientes mal clasificados.
5. **Curva ROC** para comparar thresholds y AUC.

DESAFÍO 1 - Grid Search

In [0]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": [2, 5, 10],
}

with mlflow.start_run(run_name="GridSearch - Random Forest") as run:
    grid_search = GridSearchCV(
        RandomForestClassifier(random_state=42),
        param_grid,
        cv=5,
        scoring="recall",  # priorizamos recall, la métrica clave en este problema médico
        return_train_score=True,
    )
    grid_search.fit(x_train_pr, y_train)

    best_rf = grid_search.best_estimator_
    test_preds_grid = best_rf.predict(x_test_pr)

    mlflow.log_metric("best_cv_recall", grid_search.best_score_)
    mlflow.log_metric("test_recall_best_model", recall_score(y_test, test_preds_grid))
    mlflow.log_metric("test_precision_best_model", precision_score(y_test, test_preds_grid))
    mlflow.log_metric("test_f1_best_model", f1_score(y_test, test_preds_grid))

print("Mejores parámetros:", grid_search.best_params_)
print(f"CV recall: {grid_search.best_score_:.3f}")
print(f"Test recall: {recall_score(y_test, test_preds_grid):.3f}")
print(f"Test precision: {precision_score(y_test, test_preds_grid):.3f}")

cv_res = pd.DataFrame(grid_search.cv_results_)
top5 = cv_res.sort_values("rank_test_score").head(5)[
    ["params", "mean_test_score", "std_test_score", "mean_train_score"]
]
display(top5)

DESAFÍO 2 - Curva ROC

In [0]:
from sklearn.metrics import roc_curve, auc

# Probabilidades de la clase positiva (enfermedad), no la predicción binaria directa
y_proba = forest_clf.predict_proba(x_test_pr)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc_curve = auc(fpr, tpr)

with mlflow.start_run(run_name="Curva ROC - Random Forest"):
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC (AUC = {roc_auc_curve:.3f})")
    ax.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Azar (AUC = 0.5)")
    ax.set_xlabel("Tasa de Falsos Positivos (1 - Especificidad)")
    ax.set_ylabel("Tasa de Verdaderos Positivos (Recall / Sensibilidad)")
    ax.set_title("Curva ROC - Modelo final (test)")
    ax.legend(loc="lower right")
    plt.show()

    mlflow.log_metric("roc_auc_curve", roc_auc_curve)
    mlflow.log_figure(fig, "roc_curve.png")

print(f"AUC: {roc_auc_curve:.3f}")

DESAFÍO 3 - Importancia de características

In [0]:
# Nombres de las columnas después del ColumnTransformer: las numéricas mantienen su
# nombre, y el OneHotEncoder expande cada categórica en una columna por categoría.
num_names = num_attr
cat_names = full_pipeline.named_transformers_["cat"].get_feature_names_out(cat_attr)
feature_names = list(num_names) + list(cat_names)

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": forest_clf.feature_importances_,
}).sort_values("importance", ascending=False)

display(feature_importance)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=feature_importance.head(15), x="importance", y="feature", ax=ax)
ax.set_title("Importancia de características - Random Forest")
plt.tight_layout()
plt.show()

# Interpretación:
# Las variables más importantes según Random Forest son thal_2 (0.104), thalach
# (0.097), cp_typical angina (0.089), oldpeak (0.083), chol (0.070) y age (0.066).
#
# Esto coincide bastante bien con lo que ya habíamos visto en la matriz de correlación
# del análisis exploratorio: 'thalach' (correlación 0.42 con target) y 'oldpeak'
# (-0.43) estaban entre las variables con mayor correlación lineal, y ahora aparecen
# también entre las más importantes para el modelo. 'thal' también correlacionaba
# fuerte (-0.34), y aquí aparece desglosada en sus categorías (thal_2 en el puesto 1,
# thal_3 en el puesto 8), lo que en conjunto la convierte en una de las variables más
# influyentes del modelo.
#
# El resultado más interesante es 'cp_typical angina' en el puesto 3: confirma la
# observación contraintuitiva del countplot de tipo de dolor de pecho, donde vimos que
# la mayoría de pacientes con 'typical angina' NO tenían enfermedad. El modelo ha
# aprendido esa señal, aunque no fuera la relación "esperada" a primera vista.
#
# 'chol' (colesterol) aparece en el puesto 5 pese a tener una correlación lineal casi
# nula con target en el heatmap (-0.07). Esto tiene sentido: la correlación de Pearson
# solo mide relaciones lineales, mientras que Random Forest puede aprovechar el
# colesterol en combinación con otras variables (por ejemplo, junto con la edad o el
# tipo de dolor de pecho) de forma no lineal, algo que un análisis de correlación
# simple no puede detectar.
#
# En conjunto, las variables clínicas relacionadas con la respuesta del corazón al
# esfuerzo (thalach, oldpeak, cp, ca) dominan la importancia, más que las variables
# demográficas o de laboratorio en reposo (age, chol, trestbps), lo cual es coherente
# desde el punto de vista médico: cómo responde el corazón durante el ejercicio es un
# indicador más directo de enfermedad cardiovascular que una medición aislada en reposo.

DESAFÍO 4 - Comparar más modelos

In [0]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

modelos = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
}

resultados_modelos = []
for nombre, modelo in modelos.items():
    with mlflow.start_run(run_name=f"Compare - {nombre}"):
        cv_recall = cross_val_score(modelo, x_train_pr, y_train, cv=5, scoring="recall")
        cv_preds = cross_val_predict(modelo, x_train_pr, y_train, cv=5)

        fila = {
            "modelo": nombre,
            "cv_recall_mean": round(cv_recall.mean(), 3),
            "cv_recall_std": round(cv_recall.std(), 3),
            "cv_precision": round(precision_score(y_train, cv_preds), 3),
            "cv_f1": round(f1_score(y_train, cv_preds), 3),
            "cv_roc_auc": round(roc_auc_score(y_train, cv_preds), 3),
        }
        mlflow.log_param("model_type", nombre)
        mlflow.log_metrics({k: v for k, v in fila.items() if k != "modelo"})
        resultados_modelos.append(fila)

df_modelos = pd.DataFrame(resultados_modelos).sort_values("cv_recall_mean", ascending=False)
display(df_modelos)
